<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/fundamentos/notebooks/c1_l7.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C1-L7 · Ficha de BTC en una página
Calcula los números de tu ficha (retorno, volatilidad, caída máxima) sobre 90 días de BTC. Pandas + matplotlib.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

CSV = 'c1_l7_btc.csv'
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/fundamentos/data/' + CSV
try:
    df = pd.read_csv(URL, parse_dates=['date'])
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data') / CSV, Path('data') / CSV, Path(CSV)]:
        if cand.exists():
            df = pd.read_csv(cand, parse_dates=['date'])
            break
    print('Fuente: local')
print(f"días={len(df)}  {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()}")
print(f"cierre inicial={df['close'].iloc[0]:.2f}  cierre final={df['close'].iloc[-1]:.2f}")

In [ ]:
rets = np.log(df['close'] / df['close'].shift(1)).dropna()
total = df['close'].iloc[-1] / df['close'].iloc[0] - 1
vol_anual = rets.std(ddof=0) * np.sqrt(365)
roll_max = df['close'].cummax()
drawdown = df['close'] / roll_max - 1
mdd = drawdown.min()
print(f'retorno total simple: {total:+.2%}')
print(f'suma de logs: {rets.sum():+.4f}  (log total: {np.log(df["close"].iloc[-1]/df["close"].iloc[0]):+.4f})')
print(f'volatilidad anualizada: {vol_anual:.1%}')
print(f'caída máxima: {mdd:.2%}  (máx {roll_max[drawdown.idxmin()]:.0f} → mín {df.loc[drawdown.idxmin(), "close"]:.0f})')
print(f'mejor día: {rets.max():+.2%}  peor día: {rets.min():+.2%}')

## Del gráfico a la tesis (nunca al revés)
Grafica primero, escribe después. Marca el máximo y la zona de caída máxima: si el gráfico contradice tu tesis, manda el gráfico.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df['date'], df['close'], c='#5eead4', lw=1.5)
ax1.fill_between(df['date'], df['close'], roll_max, where=(drawdown < 0),
                   color='#f59e0b', alpha=0.25, label='caída desde máximo')
ax1.set_title('BTC 90 días: +0,7% al cierre con un −10,7% en el medio')
ax1.set_ylabel('cierre USD')
ax1.legend()
ax2.fill_between(df['date'], drawdown * 100, 0, color='#f59e0b', alpha=0.6)
ax2.set_ylabel('drawdown %')
ax2.set_xlabel('fecha')
fig.tight_layout()
fig

In [ ]:
# Chequeos automáticos
assert len(df) == 90, 'deben ser 90 días'
assert str(df['date'].iloc[0].date()) == '2024-01-01'
assert str(df['date'].iloc[-1].date()) == '2024-03-30'
assert abs(df['close'].iloc[0] - 42000.0) < 1e-6
assert abs(total - 0.0072) < 1e-3, 'el retorno total debe rondar +0,72%'
assert abs(rets.sum() - np.log(df['close'].iloc[-1] / df['close'].iloc[0])) < 1e-9, 'los logs deben sumar el total'
assert 0.15 < vol_anual < 0.35, 'la vol anual debe rondar ~24%'
assert abs(mdd - (-0.1069)) < 0.005, 'la caída máxima debe rondar −10,7%'
print('OK: ficha con números verificados')